# BARA-Net on TS-SatFire (Colab, end-to-end)

Run the whole pipeline on real [TS-SatFire](https://www.kaggle.com/datasets/z789456sx/ts-satfire) data:

1. clone this repo and import the package,
2. download a small slice of the Kaggle dataset (no Kaggle credentials needed),
3. audit the BA label band,
4. build 3-day windows with cumulative thermal preprocessing,
5. forward pass, short training run, evaluation and full-scene inference.

A T4 GPU is enough. Only a few fires are downloaded, not the full 76 GB dataset.


In [ ]:
# 1. Environment + repo
import os, subprocess, sys

REPO_URL = "https://github.com/AneKazek/baranet_wildfire-segmentation.git"
GITHUB_TOKEN = ""  # set a token only if the repo is private

if not os.path.exists("/content/baranet/src"):
    url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", url, "/content/baranet"], check=True)
sys.path.insert(0, "/content/baranet/src")

import torch
from baranet import Trainer, build_model, load_model_config, load_train_config, morphological_gradient

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())


## 2. Download a few fires

TS-SatFire layout: `<fire_id>/VIIRS_Day/<date>_VIIRS_Day.tif` plus the matching
`VIIRS_Night` file. Date lists are hard-coded here; list other fires with
`kaggle datasets files -d z789456sx/ts-satfire`.


In [ ]:
DATASET = "z789456sx/ts-satfire"
RAW = "/content/ts_satfire"

if subprocess.run(["which", "kaggle"], capture_output=True).returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)

# Fires kept because their band-7 (BA) label is actually annotated.
FIRE_DAYS = {
    "21889992": ["2018-07-16","2018-07-17","2018-07-18","2018-07-19","2018-07-20","2018-07-21","2018-07-22","2018-07-23"],
    "21998023": ["2018-08-07","2018-08-08","2018-08-09","2018-08-10","2018-08-11","2018-08-12","2018-08-13","2018-08-14"],
}


def fetch(remote):
    _, fire, kind, name = remote.split("/")
    dest = os.path.join(RAW, fire, kind)
    os.makedirs(dest, exist_ok=True)
    local = os.path.join(dest, name)
    if not os.path.exists(local):
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", DATASET, "-f", remote, "-p", dest, "--quiet"],
            check=True,
        )
    if not os.path.exists(local):
        raise FileNotFoundError(f"kaggle download produced no file at {local}")
    return local


for fire, dates in FIRE_DAYS.items():
    for date in dates:
        fetch(f"ts-satfire/{fire}/VIIRS_Day/{date}_VIIRS_Day.tif")
        fetch(f"ts-satfire/{fire}/VIIRS_Night/{date}_VIIRS_Night.tif")

print({fire: len(dates) for fire, dates in FIRE_DAYS.items()}, "days cached in", RAW)


## 3. Band audit

`VIIRS_Day` has 8 bands whose descriptions are `b1, b2, b3, b4, b5, m11, af, first`.
Bands 0-5 are the day inputs, band 6 is the AF label, band 7 (`first`) is the BA
label: **finite = burned, NaN = not burned**. An entirely NaN band means the
date has no BA annotation and must not be trained on as an all-negative target.


In [ ]:
import numpy as np, rasterio


def read_day(fire, date):
    with rasterio.open(f"{RAW}/{fire}/VIIRS_Day/{date}_VIIRS_Day.tif") as src:
        return src.read().astype(np.float32)


for fire, dates in FIRE_DAYS.items():
    counts = [int(np.isfinite(read_day(fire, date)[7]).sum()) for date in dates]
    print(fire, "band-7 finite pixels per date:", counts)


## 4. Windows and targets

Eight logical input channels: `Day b1,b2,b3,b4,b5,m11` + `Night I4,I5`.
Following the TS-SatFire BA recipe, I4/I5 are replaced by their pixel-wise
maximum over the event history, and the BA target is the cumulative union of
annotated burned pixels up to the target date.


In [ ]:
MEAN = np.array([18.76488, 27.441864, 20.584806, 305.99478,
                 294.31738, 14.625097, 276.4207, 275.16766], np.float32)
STD = np.array([15.911591, 14.879259, 10.832616, 21.761852,
                24.703484, 9.878246, 40.64329, 40.7657], np.float32)


def read_night(fire, date):
    path = f"{RAW}/{fire}/VIIRS_Night/{date}_VIIRS_Night.tif"
    if not os.path.exists(path):
        return None
    with rasterio.open(path) as src:
        return src.read().astype(np.float32)[-2:]  # Night I4, I5


def fire_sequence(fire, dates):
    """frames [D,8,H,W], cumulative BA masks [D,H,W], annotation availability [D]."""
    frames, masks, annotated = [], [], []
    cmax = acc = None
    for date in dates:
        day = read_day(fire, date)
        night = read_night(fire, date)
        if night is None:
            night = np.zeros((2, *day.shape[1:]), np.float32)
        raw = np.concatenate([np.nan_to_num(day[:6]), np.nan_to_num(night)])
        cmax = raw if cmax is None else np.maximum(cmax, raw)
        frames.append(np.concatenate([raw[:3], cmax[3:5], raw[5:6], cmax[6:8]]))
        labelled = np.isfinite(day[7])
        acc = labelled if acc is None else (acc | labelled)
        masks.append(acc.astype(np.float32))
        annotated.append(bool(labelled.any()))
    return np.stack(frames), np.stack(masks), np.array(annotated)


## 5. Patch sampling

Full-scene tiling with stride 128: every patch that contains burned pixels is a
candidate (capped per window), plus the centre patch so negative patches are
represented too.


In [ ]:
PATCH, STRIDE, MAX_POS = 256, 128, 4
rng = np.random.default_rng(0)


def grid(size):
    return sorted(set(list(range(0, size - PATCH + 1, STRIDE)) + [size - PATCH]))


def make_items(fire, dates):
    frames, masks, annotated = fire_sequence(fire, dates)
    height, width = frames.shape[-2:]
    items = []
    for t in range(2, len(dates)):
        if not annotated[t]:
            continue
        window = (frames[t - 2:t + 1] - MEAN[None, :, None, None]) / STD[None, :, None, None]
        target = masks[t]
        positive = [
            (row, col)
            for row in grid(height)
            for col in grid(width)
            if target[row:row + PATCH, col:col + PATCH].any()
        ]
        if len(positive) > MAX_POS:
            picks = sorted(rng.choice(len(positive), MAX_POS, replace=False))
            positive = [positive[i] for i in picks]
        centre = (height // 2 - PATCH // 2, width // 2 - PATCH // 2)
        for row, col in positive + [centre]:
            items.append({
                "x": window[:, :, row:row + PATCH, col:col + PATCH].astype(np.float32),
                "y": target[row:row + PATCH, col:col + PATCH][None].astype(np.float32),
                "fire": fire,
            })
    return items


items = {fire: make_items(fire, dates) for fire, dates in FIRE_DAYS.items()}
print({fire: len(p) for fire, p in items.items()}, "patches")


In [ ]:
# 6. Torch dataset that matches the batch contract in the README
from torch.utils.data import DataLoader, Dataset


class PatchDataset(Dataset):
    def __init__(self, patches):
        self.patches = patches

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, index):
        patch = self.patches[index]
        target = torch.from_numpy(patch["y"])
        return {
            "inputs": torch.from_numpy(patch["x"]),
            "ba_target": target,
            "boundary_target": morphological_gradient(target[None])[0],
            "valid_mask": torch.ones_like(target, dtype=torch.bool),
            "fire_id": patch["fire"],
        }


fires = sorted(items)
train_loader = DataLoader(
    PatchDataset([p for fire in fires[:-1] for p in items[fire]]), batch_size=2, shuffle=True
)
val_loader = DataLoader(PatchDataset(items[fires[-1]]), batch_size=2)
print("train patches", len(train_loader.dataset), "| val patches", len(val_loader.dataset))


## 6. Model, training, evaluation

The repo ships the architecture, losses, metrics and training loop; this
notebook only supplies the data. Two epochs are enough to prove the loop runs
on real data — this is not a publication run.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_model(load_model_config("/content/baranet/configs/model/baranet.yaml")).to(device)
with torch.no_grad():
    out = model(train_loader.dataset[0]["inputs"][None].to(device))
print({k: None if v is None else tuple(v.shape) for k, v in out.items()})
print("params (M)", round(sum(p.numel() for p in model.parameters()) / 1e6, 2))

cfg = load_train_config("/content/baranet/configs/train/default.yaml")
cfg.epochs = 2
cfg.batch_size = 2
cfg.gradient_accumulation_steps = 1
cfg.warmup_epochs = 1
cfg.early_stopping_patience = 5

trainer = Trainer(model, train_loader, val_loader, cfg,
                  out_dir="/content/artifacts/colab_demo", seed=17)
result = trainer.fit()
print("best macro F1", round(result["best_macro_f1"], 4), "at epoch", result["best_epoch"])


In [ ]:
# 7. Validation metrics + full-scene inference on the held-out fire
from baranet import predict_scene

print({k: round(v, 4) for k, v in trainer.evaluate().items()})

fire = fires[-1]
frames, masks, _ = fire_sequence(fire, FIRE_DAYS[fire])
t = len(FIRE_DAYS[fire]) - 1
scene = (frames[t - 2:t + 1] - MEAN[None, :, None, None]) / STD[None, :, None, None]
prob = predict_scene(trainer.model, torch.from_numpy(scene), patch_size=256, stride=192)
truth = torch.from_numpy(masks[t])
print("scene", tuple(prob.shape), "| mean prob", round(float(prob.mean()), 4),
      "| burned px (GT)", int(truth.sum()))

checkpoints = sorted(os.listdir("/content/artifacts/colab_demo/checkpoints"))
print("checkpoints", checkpoints)


## 8. Next steps

* Precompute and cache `sp_map` (SLIC) per patch — recomputing it every forward
  pass dominates CPU time.
* Train on the full BA pool with the seeded 3-run protocol from
  `configs/train/default.yaml`, select the threshold on validation only.
* `predict_scene` gives full-scene probabilities; the reported protocol and
  caveats are in `prd.md`.
